# Protected Areas

This notebook loads the WDPA polygons and assigns weights depending on the IUCN category. This assignement should reflect higher sensitivity for some categories than for others. Furthermore,the notebook filters the data so only terrestrial and coastal protected areas are included. Finally, the notebook rasterizes the polygons using bii layer as reference grid, overlays them on country boundaries and creates: 
- a raster (with bii as reference raster)
- a map figure (`OUT_PNG`)
  
## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `wdpa1.gpkg`
- `wdpa2.gpkg`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)



In [ ]:
# Configuration (edit these paths / settings)
WDPA1_GPKG = 'wdpa1.gpkg'
WDPA2_GPKG = 'wdpa2.gpkg'
WDPA_SMALL_GPKG = 'wdpa_small.gpkg'
BII_5000M_TIF = 'sensitivity/biodiversity_intactness/bii_5000m.tif'
WDPA_RASTERIZED_TIF = 'wdpa_rasterized.tif'
WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG = 'World_Countries_(Generalized)_8414823838130214587.gpkg'
WDPA_5000M_TIF = 'wdpa_5000m.tif'

# Tip: keep data files out of the repo (use .gitignore) or use Git LFS/DVC for large files.


In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from rasterio.enums import Resampling
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from rasterio.windows import Window


In [ ]:
#load wdpa data
wdpa1= gpd.read_file(WDPA1_GPKG)
wdpa2= gpd.read_file(WDPA2_GPKG)

In [ ]:
wdpa1.head()

In [ ]:
# align to same CRS (reproject wdpa2 to wdpa1 CRS)
wdpa2 = wdpa2.to_crs(wdpa1.crs)

# Merge into one GeoDataFrame
wdpa_merged = gpd.GeoDataFrame(
    pd.concat([wdpa1, wdpa2], ignore_index=True),
    crs=wdpa1.crs
)


In [ ]:
#create weights dictionary
weights = {
    "Ia": 1.00,
    "Ib": 1.00,      
    "II": 0.90,
    "III": 0.90,     
    "IV": 0.70,
    "V": 0.40,
    "VI": 0.40,     
    "Not assigned": 0.50,
    None: 0.50,      
    "": 0.50
}


In [ ]:
# Assign weights using map()
wdpa_merged["weights"] = wdpa_merged["IUCN_CAT"].map(weights).fillna(0.5)

In [ ]:
#filter for only terrestrial and coastal protected areas
wdpa_merged = wdpa_merged[wdpa_merged["MARINE"].isin(["0", "1"])]

In [ ]:
wdpa_merged.head()

In [ ]:
print(wdpa_merged.crs)


In [ ]:
#simplify geometry
wdpa_small = wdpa_merged[["geometry", "weights"]].copy()
wdpa_small["geometry"] = wdpa_small.geometry.simplify(0.03, preserve_topology=True)


In [ ]:
#save as gpkg
wdpa_small.to_file(WDPA_SMALL_GPKG, driver="GPKG")

In [ ]:
# rasterize proteceted areas variable using bii as reference raster

#paths
ref = BII_5000M_TIF #reference raster
out_path = WDPA_RASTERIZED_TIF

# Open reference raster as the template grid
with rasterio.open(ref) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    out_shape = (src.height, src.width)

    # reproject polygons to match raster CRS
    gdf_r = wdpa_small.to_crs(crs)
    
    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):    
            win_transform = rasterio.windows.transform(window, transform)
        
            window_raster = rasterize(
                    [(geom, value) for geom, value in zip(gdf_r.geometry, gdf_r.weights)],
                    out_shape=(window.height, window.width),  
                    transform=win_transform,
                    fill=-9999,
                    dtype="float32"
                )
    
            dst.write(window_raster, 1, window=window)

print("Saved:", out_path)


In [ ]:
#add country boundaries

#paths
wdpa_path = WDPA_RASTERIZED_TIF  # your existing raster
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_path = WDPA_5000M_TIF

# Open WDPA raster as the template grid
with rasterio.open(wdpa_path) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    src_nodata = src.nodata 

    # Read countries and project to raster CRS
    world = gpd.read_file(countries_path).to_crs(crs)

    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):
            wdpa = src.read(1, window=window).astype("float32")
            win_transform = rasterio.windows.transform(window, transform)
            
            country_mask = rasterize(
                [(geom, 1) for geom in world.geometry],
                out_shape=(window.height, window.width),
                transform=win_transform,
                fill=0,
                dtype="uint8"
            )

            # Create output for this window
            out = np.full((window.height, window.width), -9999, dtype="float32")
            inside = country_mask == 1

            # inside countries: treat -9999 (no data) in kba as 0 (no kba)
            wdpa_inside = wdpa.copy()
            wdpa_inside[(wdpa_inside == -9999)] = 0.0
            out[inside] = wdpa_inside[inside]

            dst.write(out, 1, window=window)

print("Saved:", out_path)


In [ ]:
#Plot

#paths
raster_path = WDPA_5000M_TIF
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_png = "wdpa.png"


# load raster
with rasterio.open(raster_path) as src:
    arr = src.read(1)
    bounds = src.bounds
    crs = src.crs


# Mask -9999 (nodata) + treat 0 as transparent 
masked = np.ma.masked_where((arr == 0) | (arr == -9999), arr)
#define nodata
nodata = (arr == -9999)

# Unique values actually used in the weights
values = sorted(set(weights.values()))   # [0.4, 0.5, 0.7, 0.9, 1.0]

# Create bin edges halfway between class values
midpoints = [(values[i] + values[i + 1]) / 2 for i in range(len(values) - 1)]
class_bounds = [values[0] - 0.05] + midpoints + [values[-1] + 0.05]

# Discrete colors: low -> high
colors = [
    "#E8F5EC",  # 0.40
    "#B7E4BE",  # 0.50
    "#7FD082",  # 0.70
    "#53C256",  # 0.90
    "#1B7F3A"   # 1.00
]

cmap = ListedColormap(colors)
cmap.set_bad(color="none")

norm = BoundaryNorm(class_bounds, cmap.N)

# load countries
world = gpd.read_file(countries_path)

#open country boundaries gpkg and drop Antarctica 
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"].copy()


#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

#plot country boundaries, set color and linewidth
world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#B9C0C8",
    linewidth=0.35,
    zorder=1
)

# Raster overlay
raster = ax.imshow(
    masked,
    cmap=cmap,
    norm=norm,
    extent=[raster_bounds.left, raster_bounds.right, raster_bounds.bottom, raster_bounds.top],
    interpolation="nearest",
    alpha=0.95,
    zorder=2
)

# Country boundaries on top
world.boundary.plot(
    ax=ax,
    color="#695C5A",
    linewidth=0.3,
    zorder=3
)

# title
ax.set_title(
    "Protected areas",
    fontsize=18,
    fontweight="semibold",
    pad=14
)

ax.set_axis_off()

cbar = plt.colorbar(
    raster,
    ax=ax,
    boundaries=class_bounds,
    ticks=values,
    spacing="proportional",
    fraction=0.03,
    pad=0.02
)

cbar.set_label("Protected area sensitivity score", fontsize=11, color="#2B2F36")
cbar.ax.tick_params(labelsize=10, colors="#2B2F36")
cbar.outline.set_edgecolor("#E3E6EA")
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

# numeric labels
cbar.set_ticklabels(["0.40", "0.50", "0.70", "0.90", "1.00"])

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
